In [1]:
import os
import sys
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import numpy as np
import cv2

ROOT_DIR = "D:\\PVOutputPrediction\\preprocessing\\cloud_detection"
HELPER_DIR = os.path.join(ROOT_DIR, "Helper Functions")
CSL_DIR = os.path.join(ROOT_DIR, "Clear_sky_library")
DATA_DIR = os.path.join(ROOT_DIR, "Data")

if HELPER_DIR not in sys.path:
    sys.path.append(HELPER_DIR)

from cloud_detection import cloud_detection_modified
from Sun_position import sun_position

In [2]:
print("ROOT_DIR:", ROOT_DIR)
print("HELPER_DIR exists:", os.path.exists(HELPER_DIR))
print("CSL_DIR exists:", os.path.exists(CSL_DIR))
print("DATA_DIR exists:", os.path.exists(DATA_DIR))

print("\nHelper files:")
print(os.listdir(HELPER_DIR))

print("\nClear sky files:")
print(os.listdir(CSL_DIR))

print("\nData files:")
print(os.listdir(DATA_DIR))

ROOT_DIR: D:\PVOutputPrediction\preprocessing\cloud_detection
HELPER_DIR exists: True
CSL_DIR exists: True
DATA_DIR exists: True

Helper files:
['cloud_detection.py', 'Relative_op_func.py', 'Sun_position.py', '__pycache__']

Clear sky files:
['Creating_CSL_library.ipynb', 'CSL_env', 'csl_images_2017.npy', 'csl_images_2018.npy', 'csl_images_2019.npy', 'csl_pv_2017.npy', 'csl_pv_2018.npy', 'csl_pv_2019.npy', 'csl_sun_center_2017.npy', 'csl_sun_center_2018.npy', 'csl_sun_center_2019.npy', 'csl_times_2017.npy', 'csl_times_2018.npy', 'csl_times_2019.npy']

Data files:
['2017_2019_images_pv_processed.hdf5', 'times_test.npy', 'times_trainval.npy', 'video_prediction_dataset (1).hdf5']


# Load Data

In [3]:
# data_path = os.path.join(DATA_DIR, "2017_2019_images_pv_processed.hdf5")

# # load testing timestamps
# times_test = np.load(os.path.join(DATA_DIR, "times_test.npy"), allow_pickle=True)
# print("times_test.shape:", times_test.shape)

# with h5py.File(data_path, "r") as f:
#     images_log_test = f["test"]["images_log"][...]
#     pv_log_test = f["test"]["pv_log"][...]

# # process image / pv data exactly as before
# images_log_test = (images_log_test / 255.0).astype("float32")
# pv_log_test = pv_log_test.astype("float32")

# print("images_log_test.shape:", images_log_test.shape)
# print("pv_log_test.shape:", pv_log_test.shape)
# print("image min/max:", images_log_test.min(), images_log_test.max())

In [4]:
import pandas as pd

# Load the specified parquet files for trainval and test metadata
metadata_trainval_path = r"D:\PVOutputPrediction\preprocessing\data\data_forecast_64\metadata_trainval\metadata.parquet"
metadata_test_path = r"D:\PVOutputPrediction\preprocessing\data\data_forecast_64\metadata_test\metadata.parquet"

df_trainval = pd.read_parquet(metadata_trainval_path)
df_test = pd.read_parquet(metadata_test_path)

print(f"Train/Val metadata shape: {df_trainval.shape}")
print(f"Test metadata shape: {df_test.shape}")

Train/Val metadata shape: (149680, 4)
Test metadata shape: (6145, 4)


In [5]:
def vis_single_img(time_stamp, image, year):
	cloud_cover, cloud_mask_tw, cloud_mask_twf, cloud_mask_f, circ_sun_disk_mask, circ_sun_rim_mask, sun_cloudiness = cloud_detection_modified(
		time=time_stamp,
		image=image,
		clear_sky_directory_path=CSL_DIR,
		year=year,
		twbs_th=0.165,
		fixed_th=0.215,
		cloudiness_th=0.7,
		circ_outer_r=10,
		circ_inner_r=6,
		sun_cloudiness_th=0.3,
		min_cloudiness_merge=0.045,
	)
	# f, ax = plt.subplots(3, 2, figsize=(12, 12))

	# ax[0, 0].imshow(image[:, :, ::-1])
	# ax[0, 0].set_xticks([])
	# ax[0, 0].set_yticks([])
	# ax[0, 0].set_title("Original image")

	# kernel = np.ones((2, 2), np.uint8)
	# bound_cloud = cv2.morphologyEx(cloud_mask_tw.astype(np.uint8), cv2.MORPH_GRADIENT, kernel)

	# ax[0, 1].imshow(image[:, :, ::-1], interpolation="none")
	# ax[0, 1].imshow(cloud_mask_tw, interpolation="none", alpha=0.2)
	# ax[0, 1].imshow(bound_cloud, interpolation="none", alpha=0.3)
	# ax[0, 1].set_xticks([])
	# ax[0, 1].set_yticks([])
	# ax[0, 1].set_title("Tw Cloud detection result")
	# ax[0, 1].text(
	# 	0.25, 0.025,
	# 	f"Cloud fraction={cloud_cover:.2f}",
	# 	color="white",
	# 	transform=ax[0, 1].transAxes
	# )

	# bound_cloud = cv2.morphologyEx(cloud_mask_twf.astype(np.uint8), cv2.MORPH_GRADIENT, kernel)

	# ax[1, 0].imshow(image[:, :, ::-1], interpolation="none")
	# ax[1, 0].imshow(cloud_mask_twf, interpolation="none", alpha=0.2)
	# ax[1, 0].imshow(bound_cloud, interpolation="none", alpha=0.3)
	# ax[1, 0].set_xticks([])
	# ax[1, 0].set_yticks([])
	# ax[1, 0].set_title("Twf Cloud detection result")

	# bound_cloud = cv2.morphologyEx(circ_sun_rim_mask.astype(np.uint8), cv2.MORPH_GRADIENT, kernel)

	# ax[1, 1].imshow(image[:, :, ::-1], interpolation="none")
	# ax[1, 1].imshow(circ_sun_rim_mask, interpolation="none", alpha=0.2)
	# ax[1, 1].set_xticks([])
	# ax[1, 1].set_yticks([])
	# ax[1, 1].set_title(f"sun_circ_disk | sun_cloudiness={sun_cloudiness:.2f}")

	# bound_cloud = cv2.morphologyEx(cloud_mask_f.astype(np.uint8), cv2.MORPH_GRADIENT, kernel)

	# ax[2, 0].imshow(image[:, :, ::-1], interpolation="none")
	# ax[2, 0].imshow(cloud_mask_f, interpolation="none", alpha=0.2)
	# ax[2, 0].imshow(bound_cloud, interpolation="none", alpha=0.3)
	# ax[2, 0].set_xticks([])
	# ax[2, 0].set_yticks([])
	# ax[2, 0].set_title("fixed_th Cloud detection result")

	# bound_cloud = cv2.morphologyEx(cloud_mask_twf.astype(np.uint8), cv2.MORPH_GRADIENT, kernel)

	# ax[2, 1].imshow(image[:, :, ::-1], interpolation="none")
	# ax[2, 1].imshow(cloud_mask_twf, interpolation="none", alpha=0.2)
	# ax[2, 1].imshow(bound_cloud, interpolation="none", alpha=0.3)
	# ax[2, 1].set_xticks([])
	# ax[2, 1].set_yticks([])
	# ax[2, 1].set_title(f"final | cloud_cover={cloud_cover:.2f}")

	# f.tight_layout()
	# plt.show()
	return cloud_cover

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
def load_mp4_to_numpy(mp4_path, max_frames=None):
    """
    Loads an mp4 file and converts it to a numpy array of frames.
    
    Args:
        mp4_path (str): Path to the mp4 video.
        max_frames (int, optional): Maximum number of frames to load. If None, load all frames.
        
    Returns:
        frames (np.ndarray): Array of shape (num_frames, H, W, 3) with dtype=np.uint8.
    """
    cap = cv2.VideoCapture(mp4_path)
    frames = []
    count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Convert BGR (OpenCV default) to RGB for matplotlib
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
        count += 1
        if max_frames is not None and count >= max_frames:
            break
    cap.release()
    if frames:
        return np.stack(frames)
    else:
        return np.array([])

video_cloudiness = np.zeros(df_trainval.shape[0])

for i in range(df_trainval.shape[0]):
	mp4_path = f"D:/PVOutputPrediction/preprocessing/data/data_forecast_64/videos_trainval/sequence_{i:06d}.mp4"
	vid_frames = load_mp4_to_numpy(mp4_path)
	cloudiness_list = []
	for j in range(len(vid_frames)):
		cloudiness = vis_single_img(df_trainval["time"][i]+pd.Timedelta(minutes=j), vid_frames[j], df_trainval["time"][i].year)
		# print(df_trainval["time"][i])
		# print(cloudiness)
		# plt.imshow(vid_frames[j])
		# plt.show()
		cloudiness_list.append(cloudiness)
	video_cloudiness[i]= np.mean(np.array(cloudiness_list))

	if i%200==0:
		print(i)

np.save(r"D:\PVOutputPrediction\preprocessing\data\data_forecast4\metadata_trainval\video_cloudiness.npy", video_cloudiness)

df_trainval["cloudiness"] = video_cloudiness
df_trainval.to_parquet(r"D:\PVOutputPrediction\preprocessing\data\data_forecast4\metadata_trainval\metadata3.parquet", index=False)


'''idx=13000
print(df_trainval["time"][idx].year)
mp4_path = f"D:/PVOutputPrediction/preprocessing/data/data_forecast_64/videos_trainval/sequence_{idx:06d}.mp4"
vid_frames = load_mp4_to_numpy(mp4_path)
print(vis_single_img(df_trainval["time"][idx], vid_frames[0], 2017))
plt.imshow(vid_frames[0])
plt.show()'''

0
200
400
600
800
1000
1200
1400
1600
1800
2000
2200
2400
2600
2800
3000
3200
3400
3600
3800
4000
4200
4400
4600
4800
5000
5200
5400
5600
5800
6000
6200
6400
6600
6800
7000
7200
7400
7600
7800
8000
8200
8400
8600
8800
9000
9200
9400
9600
9800
10000
10200
10400
10600
10800
11000
11200
11400
11600
11800
12000
12200
12400
12600
12800
13000
13200
13400
13600
13800
14000
14200
14400
14600
14800
15000
15200
15400
15600
15800
16000
16200
16400
16600
16800
17000
17200
17400
17600
17800
18000
18200
18400
18600
18800
19000
19200
19400
19600
19800
20000
20200
20400
20600
20800
21000
21200
21400
21600
21800
22000
22200
22400
22600
22800
23000
23200
23400
23600
23800
24000
24200
24400
24600
24800
25000
25200
25400
25600
25800
26000
26200
26400
26600
26800
27000
27200
27400
27600
27800
28000
28200
28400
28600
28800
29000
29200
29400
29600
29800
30000
30200
30400
30600
30800
31000
31200
31400
31600
31800
32000
32200
32400
32600
32800
33000
33200
33400
33600
33800
34000
34200
34400
34600
34800
35000
3

In [1]:
print(df_trainval)

NameError: name 'df_trainval' is not defined

In [ ]:
for i in range(df_trainval.shape[0]):
    

In [ ]:
i = 10900
mp4_path = f"D:/PVOutputPrediction/preprocessing/data/data_forecast_64/videos_trainval/sequence_{i:06d}.mp4"
vid_frames = load_mp4_to_numpy(mp4_path)

plt.imshow(vid_frames[0])
plt.show()

print(video_cloudiness[i])

NameError: name 'times_trainval' is not defined

In [16]:
for idx in range(0, min(len(times_test_cloudy_2019), 3000), 30):
    print("idx:", idx, "| time:", times_test_cloudy_2019[idx])
    vis_single_img_2019(times_test_cloudy_2019[idx], images_log_test_cloudy_2019[idx])

idx: 0 | time: 2019-05-27 06:20:10


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\PVOutputPrediction\\preprocessing\\cloud_detection\\Clear_sky_library\\csl_images_2019.npy'

In [17]:
for i in range(292):
    idx = 10*i # + 70
    print("idx:", idx, "| time:", times_test_cloudy_2019[idx])
    vis_single_img_2019(times_test_cloudy_2019[idx], images_log_test_cloudy_2019[idx])

idx: 0 | time: 2019-05-27 06:20:10


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\PVOutputPrediction\\preprocessing\\cloud_detection\\Clear_sky_library\\csl_images_2019.npy'